# PopOut AI: unified workflow

This notebook runs a single continuous project workflow:

Board logic demonstration;

Generation of the self-play dataset and ID3 training;

Cross-validation on Iris;

Comparison of different MCTS modes;

Generation of the tournament dataset;

ID3 vs MCTS matches.

“The default parameters are made compact so that 'Run All' can be executed without manual input.”

In [13]:
from pathlib import Path
import os

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / 'workflow_support.py').exists():
    candidates = [Path.cwd(), Path.cwd() / 'final']
    for candidate in candidates:
        if (candidate / 'workflow_support.py').exists():
            os.chdir(candidate)
            NOTEBOOK_DIR = candidate
            break

from workflow_support import (
    FILES_DIR,
    board_demo,
    ensure_local_paths,
    evaluate_id3_against_specs,
    generate_selfplay_dataset,
    generate_tournament_dataset,
    iris_workflow,
    round_robin_workflow,
    specs_to_frame,
    train_popout_tree_from_csv,
    workflow_specs,
)

ROOT, FILES_DIR = ensure_local_paths()
print(f'Working directory: {ROOT}')
print(f'Files directory: {FILES_DIR}')

Working directory: C:\Users\tomas\Popout AI\final
Files directory: C:\Users\tomas\Popout AI\final\files


In [14]:
SELFPLAY_GAMES = 6
SELFPLAY_ITERATIONS = 80
MCTS_ITERATIONS = 120
ROUND_ROBIN_GAMES_PER_SIDE = 1
TOURNAMENT_GAMES_PER_SIDE = 1
ID3_VS_MCTS_GAMES_PER_SIDE = 2

SELFPLAY_DATASET = FILES_DIR / 'popout_workflow_dataset.csv'
SELFPLAY_TREE = FILES_DIR / 'popout_workflow_tree.json'
TOURNAMENT_DATASET = FILES_DIR / 'popout_tournament_workflow.csv'
TOURNAMENT_TREE = FILES_DIR / 'popout_tournament_tree.json'

## 1. Board demo

In [15]:
final_state, board_demo_df = board_demo()
for row in board_demo_df.itertuples(index=False):
    print(f'=== Step {row.step}: {row.move} ===')
    print(row.board)
    print()

board_demo_df[['step', 'move', 'winner']]

=== Step 0: initial ===
- - - - - - -
- - - - - - -
- - - - - - -
- - - - - - -
- - - - - - -
- - - - - - -
0 1 2 3 4 5 6
player_to_move=X
legal_moves=['drop_0', 'drop_1', 'drop_2', 'drop_3', 'drop_4', 'drop_5', 'drop_6']

=== Step 1: drop_0 ===
- - - - - - -
- - - - - - -
- - - - - - -
- - - - - - -
- - - - - - -
X - - - - - -
0 1 2 3 4 5 6
player_to_move=O
legal_moves=['drop_0', 'drop_1', 'drop_2', 'drop_3', 'drop_4', 'drop_5', 'drop_6']

=== Step 2: drop_1 ===
- - - - - - -
- - - - - - -
- - - - - - -
- - - - - - -
- - - - - - -
X O - - - - -
0 1 2 3 4 5 6
player_to_move=X
legal_moves=['drop_0', 'drop_1', 'drop_2', 'drop_3', 'drop_4', 'drop_5', 'drop_6', 'pop_0']

=== Step 3: drop_0 ===
- - - - - - -
- - - - - - -
- - - - - - -
- - - - - - -
X - - - - - -
X O - - - - -
0 1 2 3 4 5 6
player_to_move=O
legal_moves=['drop_0', 'drop_1', 'drop_2', 'drop_3', 'drop_4', 'drop_5', 'drop_6', 'pop_1']

=== Step 4: drop_1 ===
- - - - - - -
- - - - - - -
- - - - - - -
- - - - - - -
X O - - - - -


,step,move,winner
0,0,initial,None
1,1,drop_0,None
2,2,drop_1,None
3,3,drop_0,None
4,4,drop_1,None
5,5,pop_0,None
6,6,drop_2,None


## 2. Self-play dataset + ID3 training

In [16]:
selfplay_rows = generate_selfplay_dataset(
    SELFPLAY_DATASET,
    num_games=SELFPLAY_GAMES,
    iterations=SELFPLAY_ITERATIONS,
)
print(f'Self-play rows: {len(selfplay_rows)}')

popout_tree_info = train_popout_tree_from_csv(SELFPLAY_DATASET, SELFPLAY_TREE)
popout_tree_info

A gerar 6 jogos para o dataset...
Dataset guardado com sucesso em 'C:\Users\tomas\Popout AI\final\files\popout_workflow_dataset.csv' com 79 linhas!
Self-play rows: 79


{'tree': {'c5_r0': {'O': {'c5_r1': {'O': {'player_to_move': {'O': {'c4_r3': {'O': 'pop_5',
         'Vazio': 'drop_4'}},
       'X': {'c5_r2': {'Vazio': 'drop_5', 'X': 'drop_0'}}}},
     'Vazio': {'c3_r2': {'Vazio': {'player_to_move': {'O': {'c4_r1': {'O': 'drop_3',
           'X': 'drop_5'}},
         'X': 'drop_3'}},
       'X': 'drop_2'}},
     'X': {'c2_r3': {'Vazio': {'c1_r1': {'Vazio': {'player_to_move': {'O': 'drop_3',
           'X': {'c3_r1': {'O': 'drop_1', 'Vazio': 'drop_0'}}}},
         'X': {'player_to_move': {'O': 'drop_2',
           'X': {'c3_r2': {'Vazio': 'drop_3', 'X': 'drop_2'}}}}}},
       'X': {'c0_r3': {'Vazio': {'player_to_move': {'O': {'c5_r2': {'Vazio': 'drop_0',
             'X': 'drop_6'}},
           'X': {'c6_r0': {'O': 'drop_0', 'Vazio': 'drop_5'}}}},
         'X': {'player_to_move': {'O': 'drop_1', 'X': 'pop_2'}}}}}}}},
   'Vazio': {'c4_r1': {'O': {'c1_r3': {'O': {'player_to_move': {'O': 'drop_5',
         'X': 'drop_4'}},
       'Vazio': {'c4_r0': {'O':

## 3. Iris cross-validation

In [17]:
iris_results_df = iris_workflow(csv_path='files/iris.csv', k=5, num_bins=4, seed=42)
iris_results_df

Fold 1/5: accuracy=0.867, unknown_rate=0.033, tree=arvore_iris_fold_1.json
Fold 2/5: accuracy=0.900, unknown_rate=0.067, tree=arvore_iris_fold_2.json
Fold 3/5: accuracy=0.933, unknown_rate=0.000, tree=arvore_iris_fold_3.json
Fold 4/5: accuracy=0.933, unknown_rate=0.000, tree=arvore_iris_fold_4.json
Fold 5/5: accuracy=0.867, unknown_rate=0.067, tree=arvore_iris_fold_5.json

Resumo:
Accuracy media: 0.900
Unknown rate medio: 0.033


,fold,accuracy,unknown_rate,tree_path
0,1,0.866667,0.033333,C:\Users\tomas\Popout AI\final\files\arvore_ir...
1,2,0.900000,0.066667,C:\Users\tomas\Popout AI\final\files\arvore_ir...
2,3,0.933333,0.000000,C:\Users\tomas\Popout AI\final\files\arvore_ir...
3,4,0.933333,0.000000,C:\Users\tomas\Popout AI\final\files\arvore_ir...
4,5,0.866667,0.066667,C:\Users\tomas\Popout AI\final\files\arvore_ir...


## 4. MCTS modes

In [18]:
specs = workflow_specs(iterations=MCTS_ITERATIONS)
specs_to_frame(specs)

,name,kind,iterations,exploration_c,expansion_policy,expansion_k,rollout_policy,rollout_depth_limit,child_selection,draw_value,epsilon
0,UCT_base_120,mcts,120,1.414,all,0,random,None,uct,0.5,0.10
1,UCT_low_c_120,mcts,120,0.700,all,0,random,None,uct,0.5,0.10
2,UCT_heuristic_rollout_120,mcts,120,1.414,all,0,heuristic,None,uct,0.5,0.10
3,UCT_epsilon_greedy_120,mcts,120,1.414,all,0,epsilon_greedy,None,uct,0.5,0.15


In [7]:
round_robin_summary, round_robin_ranking_df, round_robin_matches_df = round_robin_workflow(
    specs,
    games_per_side=ROUND_ROBIN_GAMES_PER_SIDE,
    num_workers=1,
)

round_robin_ranking_df

UCT_base_120 vs UCT_low_c_120: 1.0 - 1.0, draws=0, avg_moves=21.0, avg_move_time=(UCT_base_120: 0.0280s, UCT_low_c_120: 0.0265s)
UCT_base_120 vs UCT_heuristic_rollout_120: 1.0 - 1.0, draws=0, avg_moves=7.0, avg_move_time=(UCT_base_120: 0.0378s, UCT_heuristic_rollout_120: 0.2067s)
UCT_base_120 vs UCT_epsilon_greedy_120: 0.0 - 2.0, draws=0, avg_moves=28.5, avg_move_time=(UCT_base_120: 0.0288s, UCT_epsilon_greedy_120: 0.1021s)
UCT_low_c_120 vs UCT_heuristic_rollout_120: 1.0 - 1.0, draws=0, avg_moves=13.0, avg_move_time=(UCT_low_c_120: 0.0359s, UCT_heuristic_rollout_120: 0.1664s)
UCT_low_c_120 vs UCT_epsilon_greedy_120: 0.0 - 2.0, draws=0, avg_moves=16.5, avg_move_time=(UCT_low_c_120: 0.0288s, UCT_epsilon_greedy_120: 0.1169s)
UCT_heuristic_rollout_120 vs UCT_epsilon_greedy_120: 1.0 - 1.0, draws=0, avg_moves=11.0, avg_move_time=(UCT_heuristic_rollout_120: 0.1912s, UCT_epsilon_greedy_120: 0.1460s)


,agent,score
0,UCT_epsilon_greedy_120,5.0
1,UCT_heuristic_rollout_120,3.0
2,UCT_base_120,2.0
3,UCT_low_c_120,2.0


In [8]:
round_robin_matches_df

,agent_a,agent_b,score_a,score_b,draws,games,avg_moves,avg_move_time_a,avg_move_time_b
0,UCT_base_120,UCT_low_c_120,1.0,1.0,0,2,21.0,0.028017,0.026543
1,UCT_base_120,UCT_heuristic_rollout_120,1.0,1.0,0,2,7.0,0.037826,0.206707
2,UCT_base_120,UCT_epsilon_greedy_120,0.0,2.0,0,2,28.5,0.028842,0.102092
3,UCT_low_c_120,UCT_heuristic_rollout_120,1.0,1.0,0,2,13.0,0.035878,0.166432
4,UCT_low_c_120,UCT_epsilon_greedy_120,0.0,2.0,0,2,16.5,0.028800,0.116938
5,UCT_heuristic_rollout_120,UCT_epsilon_greedy_120,1.0,1.0,0,2,11.0,0.191205,0.146003


## 5. Tournament dataset + tournament-trained ID3

In [9]:
tournament_rows = generate_tournament_dataset(
    TOURNAMENT_DATASET,
    specs,
    games_per_side=TOURNAMENT_GAMES_PER_SIDE,
)
print(f'Tournament rows: {len(tournament_rows)}')

tournament_tree_info = train_popout_tree_from_csv(TOURNAMENT_DATASET, TOURNAMENT_TREE)
tournament_tree_info

A correr torneio 1/1: workflow_tournament com 4 agentes.
Dataset guardado com sucesso em 'C:\Users\tomas\Popout AI\final\files\popout_tournament_workflow.csv' com 169 linhas!
Tournament rows: 169


{'tree': {'c1_r1': {'O': {'c5_r1': {'O': {'c5_r3': {'O': {'player_to_move': {'O': 'pop_5',
         'X': 'drop_1'}},
       'Vazio': 'drop_5'}},
     'Vazio': {'player_to_move': {'O': {'c5_r0': {'O': {'c4_r0': {'Vazio': 'drop_3',
           'X': 'drop_5'}},
         'Vazio': 'drop_5'}},
       'X': {'c5_r0': {'O': {'c3_r2': {'O': 'drop_4', 'Vazio': 'drop_3'}},
         'Vazio': 'drop_1'}}}},
     'X': {'c3_r3': {'Vazio': 'drop_3', 'X': 'drop_2'}}}},
   'Vazio': {'c1_r0': {'O': {'c0_r0': {'O': {'player_to_move': {'O': 'drop_2',
         'X': 'drop_6'}},
       'Vazio': {'c5_r1': {'Vazio': {'c2_r3': {'O': 'drop_1',
           'Vazio': {'player_to_move': {'O': {'c3_r2': {'Vazio': 'drop_1',
               'X': 'drop_2'}},
             'X': 'drop_2'}}}},
         'X': 'drop_1'}},
       'X': 'drop_1'}},
     'Vazio': {'c5_r0': {'O': {'c0_r1': {'O': {'player_to_move': {'O': {'c2_r0': {'Vazio': 'drop_0',
             'X': 'drop_6'}},
           'X': 'drop_2'}},
         'Vazio': {'c3_r2': {'O

## 6. ID3 vs MCTS matches

In [10]:
id3_summary_df, id3_games_df = evaluate_id3_against_specs(
    tournament_tree_info['tree'],
    specs,
    games_per_side=ID3_VS_MCTS_GAMES_PER_SIDE,
    fallback_iterations=60,
)
id3_summary_df

,opponent,games,id3_score,mcts_score,draws
0,UCT_base_120,4,1.0,3.0,0
1,UCT_low_c_120,4,0.0,4.0,0
2,UCT_heuristic_rollout_120,4,3.0,1.0,0
3,UCT_epsilon_greedy_120,4,1.0,3.0,0


In [11]:
id3_games_df

,game,id3_side,mcts_side,winner,moves,opponent
0,1,X,O,ID3,13,UCT_base_120
1,2,X,O,UCT_base_120,22,UCT_base_120
2,3,O,X,UCT_base_120,19,UCT_base_120
3,4,O,X,UCT_base_120,21,UCT_base_120
4,1,X,O,UCT_low_c_120,8,UCT_low_c_120
5,2,X,O,UCT_low_c_120,18,UCT_low_c_120
6,3,O,X,UCT_low_c_120,7,UCT_low_c_120
7,4,O,X,UCT_low_c_120,9,UCT_low_c_120
8,1,X,O,ID3,11,UCT_heuristic_rollout_120
9,2,X,O,UCT_heuristic_rollout_120,12,UCT_heuristic_rollout_120


## 7. Output files

After Run All, the main artifacts are located in final/files/."

In [12]:
sorted(path.name for path in FILES_DIR.iterdir())

['arvore_iris_fold_1.json',
 'arvore_iris_fold_2.json',
 'arvore_iris_fold_3.json',
 'arvore_iris_fold_4.json',
 'arvore_iris_fold_5.json',
 'arvore_tour_adv.json',
 'arvore_tour_base.json',
 'arvore_tour_craz_1.json',
 'arvore_tour_craz_2_perturbed.json',
 'iris.csv',
 'popout_tournament_tree.json',
 'popout_tournament_workflow.csv',
 'popout_workflow_dataset.csv',
 'popout_workflow_tree.json',
 'tour_adv_1.csv',
 'tour_base_1.csv',
 'tour_craz_1.csv',
 'tour_craz_2_perturbed.csv']